# 04 – Attention: come le parole si guardano tra loro

In questo notebook:
1. calcoliamo l'attention a mano, con numeri piccoli;
2. guardiamo l'attention di un vero modello;
3. misuriamo perché il contesto costa.

Per la parte 2 conviene la GPU: **Runtime → Cambia tipo di runtime → T4 GPU**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def heatmap(m, etichette, titolo):
    plt.figure(figsize=(6, 5))
    plt.imshow(m, cmap='Blues')
    plt.xticks(range(len(etichette)), etichette, rotation=45)
    plt.yticks(range(len(etichette)), etichette)
    plt.colorbar(); plt.title(titolo); plt.show()

## 1. Attention a mano

Ogni parola è un vettore (**embedding**). Per l'attention, da ogni vettore si ricavano tre vettori:
- **Query**: che cosa cerco?
- **Key**: che cosa offro?
- **Value**: che cosa passo, se vengo scelta?

Si ottengono moltiplicando l'embedding per tre matrici: **Wq, Wk, Wv**.
In un modello vero sono imparate in addestramento; qui sono casuali.

In [ ]:
parole = ['Il', 'gatto', 'è', 'molto', 'stanco']
d = 4                                    # dimensione degli embedding (nei modelli veri: migliaia)
rng = np.random.default_rng(0)

E = rng.normal(size=(len(parole), d))    # un embedding per parola
Wq, Wk, Wv = (rng.normal(size=(d, d)) for _ in range(3))

Q, K, V = E @ Wq, E @ Wk, E @ Wv
print('Q ha forma', Q.shape)

### Punteggi

Ogni Query si confronta con ogni Key (prodotto scalare). Si ottiene una tabella **n × n**: ogni parola contro ogni parola.
Si divide per √d per tenere i numeri piccoli.

In [ ]:
punteggi = Q @ K.T / np.sqrt(d)
print(punteggi.round(2))

### Softmax

Trasforma ogni riga in percentuali che sommano a 1: quanto ogni parola “guarda” le altre.

In [ ]:
def softmax(m):
    m = np.exp(m - m.max(axis=1, keepdims=True))
    return m / m.sum(axis=1, keepdims=True)

attention = softmax(punteggi)
print(attention.round(2))
print('Somma di ogni riga:', attention.sum(axis=1))
heatmap(attention, parole, 'Attention con pesi casuali')

### Risultato

Ogni parola diventa una media dei Value delle altre, pesata dall'attention.
Ora “stanco” contiene un po' di informazione su “gatto”.

In [ ]:
uscita = attention @ V
print('Prima:', E.shape, ' Dopo:', uscita.shape)

Tutto qui: tre moltiplicazioni di matrici e una softmax. Un transformer ripete questo blocco
in decine di **teste** parallele e decine di **strati**.

## 2. L'attention di un modello vero

Usiamo di nuovo Qwen2.5 0.5B e chiediamo di restituire l'attention.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # evita il download che in Colab a volte si blocca
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

nome = 'Qwen/Qwen2.5-0.5B-Instruct'
dispositivo = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(nome)
modello = AutoModelForCausalLM.from_pretrained(nome, torch_dtype='auto',
                                               attn_implementation='eager').to(dispositivo)
cfg = modello.config
print('Strati:', cfg.num_hidden_layers, ' Teste per strato:', cfg.num_attention_heads)

In [ ]:
frase = 'Il gatto non salta sul tavolo perché è stanco'
ids = tokenizer(frase, return_tensors='pt').input_ids.to(dispositivo)
token = [tokenizer.decode([i]) for i in ids[0]]

with torch.no_grad():
    out = modello(ids, output_attentions=True)

print(len(out.attentions), 'strati, ognuno con forma', tuple(out.attentions[0].shape))

In [ ]:
strato, testa = 12, 0
a = out.attentions[strato][0, testa].float().cpu().numpy()
heatmap(a, token, f'Strato {strato}, testa {testa}')

La tabella è triangolare: ogni token può guardare **solo i token precedenti**, perché il modello genera da sinistra a destra.

**Prova tu:** cambia `strato` (0–23) e `testa` (0–13). Le teste si specializzano: alcune guardano la parola
precedente, altre il primo token, altre i riferimenti.

Cerchiamo la testa in cui “stanco” guarda di più “gatto”.

In [ ]:
spazi = tokenizer(frase, return_offsets_mapping=True)['offset_mapping']

def indice(parola):
    pos = frase.index(parola)
    return next(i for i, (a, b) in enumerate(spazi) if a <= pos < b)

i_stanco, i_gatto = indice('stanco'), indice('gatto')

migliore = max(((s, h, out.attentions[s][0, h, i_stanco, i_gatto].item())
                for s in range(cfg.num_hidden_layers) for h in range(cfg.num_attention_heads)),
               key=lambda r: r[2])
print(f'Strato {migliore[0]}, testa {migliore[1]}: “{token[i_stanco]}” guarda “{token[i_gatto]}” al {migliore[2]:.0%}')
a = out.attentions[migliore[0]][0, migliore[1]].float().cpu().numpy()
heatmap(a, token, f'Strato {migliore[0]}, testa {migliore[1]}')

## 3. Perché il contesto costa

La tabella di attention ha n × n caselle. Se il testo raddoppia, le caselle quadruplicano.

In [ ]:
for n in [1_000, 10_000, 100_000, 1_000_000]:
    print(f'{n:>9,} token → {n * n:>22,} confronti per testa e per strato')

In [ ]:
import time

tempi, lunghezze = [], [128, 256, 512, 1024, 2048]
for n in lunghezze:
    ids = torch.randint(0, 1000, (1, n), device=dispositivo)
    with torch.no_grad():
        modello(ids[:, :16])                     # riscaldamento
        if dispositivo == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        modello(ids, output_attentions=True)
        if dispositivo == 'cuda': torch.cuda.synchronize()
    tempi.append(time.time() - t0)
    print(f'{n:5d} token: {tempi[-1]:.2f} s')

plt.plot(lunghezze, tempi, 'o-', color='#1F3A5F')
plt.xlabel('token in ingresso'); plt.ylabel('secondi'); plt.show()

Per questo la **finestra di contesto** è una risorsa scarsa: dare al modello solo quello che serve
lo rende più veloce, più economico e spesso più preciso.

## Da ricordare

- Attention = ogni token confronta la sua Query con le Key di tutti gli altri.
- Wq, Wk, Wv sono imparate e poi fisse; la tabella n × n si ricalcola per ogni testo.
- Teste e strati moltiplicano il meccanismo.
- Il costo cresce con il quadrato della lunghezza del testo.